# 01b — Calibrazione controllata dei protocolli dendritici

Questo notebook cerca protocolli riproducibili per NMDA spike, plateau NMDA e Ca spike prima del baseline full-state. La versione 0.3 seleziona cluster lungo un singolo ramo, misura gli eventi nel centro realmente stimolato, varia esplicitamente la finestra di sincronia e conferma su un orizzonte lungo che offset e durata non siano censurati. Non modifica morfologia, meccanismi, pesi sinaptici o tolleranze del teacher.

## 1. Checkout riproducibile

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ELM_REPOSITORY = "https://github.com/Zagred47/giada.git"
ELM_REF = os.environ.get("HAYFLOW_ELM_REF", "main")
if Path("/kaggle/working").is_dir():
    NOTEBOOK_ROOT = Path("/kaggle/working")
elif Path("/content").is_dir():
    NOTEBOOK_ROOT = Path("/content")
else:
    NOTEBOOK_ROOT = Path.cwd().resolve()
WORKSPACE = NOTEBOOK_ROOT / "hayflow_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

elm_override = os.environ.get("HAYFLOW_ELM_REPO")
mounted = [Path(elm_override).expanduser()] if elm_override else []
mounted.extend([Path.cwd(), *Path.cwd().parents])
mounted_elm = next((p.resolve() for p in mounted if (p / "src" / "hayflow_teacher").is_dir()), None)
if mounted_elm is not None:
    ELM_REPO = mounted_elm
else:
    ELM_REPO = WORKSPACE / "elmneuron"
    if not (ELM_REPO / ".git").is_dir():
        run(["git", "clone", ELM_REPOSITORY, ELM_REPO])
    run(["git", "fetch", "origin", ELM_REF], cwd=ELM_REPO)
    run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ELM_REPO)
os.environ["HAYFLOW_ELM_REPO"] = str(ELM_REPO)

# Se questa cella viene rieseguita nello stesso kernel, elimina l'eventuale
# versione precedente del package dalla cache degli import.
import importlib
for module_name in tuple(sys.modules):
    if module_name == "src.hayflow_teacher" or module_name.startswith("src.hayflow_teacher."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()
print("Owned repository:", ELM_REPO)

## 2. Teacher canonico e dipendenze

In [ ]:
import shutil

TEACHER_REPOSITORY = "https://github.com/SelfishGene/neuron_as_deep_net.git"
TEACHER_COMMIT = "074c4666300a8ad246601dab179a97a6942f0f29"
teacher_override = os.environ.get("HAYFLOW_TEACHER_REPO")
sibling = ELM_REPO.parent / "neuron_as_deep_net"
TEACHER_REPO = Path(teacher_override).expanduser().resolve() if teacher_override else sibling
if not (TEACHER_REPO / ".git").is_dir():
    run(["git", "clone", TEACHER_REPOSITORY, TEACHER_REPO])
run(["git", "checkout", "--detach", TEACHER_COMMIT], cwd=TEACHER_REPO)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=TEACHER_REPO, text=True).strip() == TEACHER_COMMIT
os.environ["HAYFLOW_TEACHER_REPO"] = str(TEACHER_REPO)

run([sys.executable, "-m", "pip", "install", "--quiet", "neuron==8.2.7", "numpy", "pandas", "matplotlib", "h5py", "pyarrow", "pyyaml"])
SIMULATION_DIR = TEACHER_REPO / "L5PC_NEURON_simulation"
compiled = list(SIMULATION_DIR.rglob("libnrnmech.so"))
if not compiled:
    nrnivmodl = shutil.which("nrnivmodl") or str(Path(sys.executable).parent / "nrnivmodl")
    run([nrnivmodl, "mods"], cwd=SIMULATION_DIR)
assert list(SIMULATION_DIR.rglob("libnrnmech.so")), "MOD compilation failed"
print("Teacher ready:", TEACHER_REPO)

## 3. Stato di equilibrio e strumentazione

Ricostruiamo lo stesso teacher e misuriamo nuovamente il burn-in. Questo rende `01b` autonomo e riproducibile anche in una sessione Kaggle separata.

In [ ]:
import json
import yaml
from IPython.display import display

sys.path.insert(0, str(ELM_REPO))
from src.hayflow_data import BurnInCriteria
from src.hayflow_teacher import (
    DENDRITIC_CALIBRATION_SCHEMA_VERSION,
    DiagnosticDatasetSession,
    DendriticProtocolCalibrator,
    expected_audit_hashes,
)

BASE_CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "transition_dataset_diagnostic.yml"
CALIBRATION_CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "dendritic_protocol_calibration.yml"
base_config = yaml.safe_load(BASE_CONFIG_PATH.read_text(encoding="utf-8"))
calibration_config = yaml.safe_load(CALIBRATION_CONFIG_PATH.read_text(encoding="utf-8"))
assert calibration_config["schema_version"] == DENDRITIC_CALIBRATION_SCHEMA_VERSION
OUTPUT_DIR = NOTEBOOK_ROOT / "hayflow_dendritic_protocol_calibration"
TEACHER_STATE_DIR = OUTPUT_DIR / "_teacher_state"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(BASE_CONFIG_PATH, OUTPUT_DIR / "teacher_experiment_config.yml")
shutil.copy2(CALIBRATION_CONFIG_PATH, OUTPUT_DIR / "calibration_config.yml")

session = DiagnosticDatasetSession(
    ELM_REPO,
    TEACHER_REPO,
    output_dir=TEACHER_STATE_DIR,
    seed=base_config["runtime"]["seed"],
    expected_teacher_hashes=base_config["teacher"]["expected_source_sha256"],
)
teacher_report = session.prepare_teacher()
display(teacher_report)
assert teacher_report["segment_count"] == 642
assert teacher_report["teacher_hashes_match_audit"]

In [ ]:
burnin_config = dict(base_config["burnin"])
burnin_config["slow_mechanisms"] = tuple(burnin_config["slow_mechanisms"])
burnin_report = session.run_burn_in(BurnInCriteria(**burnin_config))
current_smoke = session.run_somatic_current_smoke_test(**base_config["somatic_current"]["smoke_test"])
somatic_calibration = session.calibrate_somatic_spike_current(
    candidate_amplitudes_na=base_config["somatic_current"]["calibration"]["candidate_amplitudes_na"]
)
display({"burnin_ms": burnin_report["burnin_duration_ms"], "current_smoke": current_smoke, "somatic_calibration": somatic_calibration})
assert burnin_report["converged"] and current_smoke["valid"] and somatic_calibration["valid"]

## 4. Sweep dendritico a pesi canonici

Le famiglie vengono esplorate dal livello meno intenso al più intenso. I protocolli NMDA usano cluster branch-aware e una sonda dinamica sul centro del cluster. Il gate richiede sia le famiglie canoniche sia la copertura robusta di NMDA spike, plateau NMDA e Ca spike su almeno due dei tre seed.

In [ ]:
experiment = dict(calibration_config["experiment"])
experiment["families"] = calibration_config["families"]
calibrator = DendriticProtocolCalibrator(
    session,
    output_dir=OUTPUT_DIR,
    sample_interval_ms=experiment["sample_interval_ms"],
)
calibration_report = calibrator.run(experiment)
display({
    "valid": calibration_report["valid"],
    "trial_count": calibration_report["trial_count"],
    "missing_required_families": calibration_report["missing_required_families"],
    "missing_required_event_kinds": calibration_report["missing_required_event_kinds"],
    "event_coverage": calibration_report["event_coverage"],
    "selected_protocols": calibration_report["selected_protocols"],
})

## 5. Conferma lunga dei protocolli selezionati

Rieseguiamo soltanto i protocolli scelti per 160 ms. Ogni evento richiesto deve tornare sotto la propria soglia di reset e i primi 35 ms devono coincidere numericamente con lo sweep breve.

In [ ]:
confirmation_report = calibrator.confirm_selected_protocols(
    duration_ms=experiment["confirmation_duration_ms"],
    overlap_tolerance=experiment["confirmation_overlap_tolerance"],
)
display({
    "valid": confirmation_report["valid"],
    "short_duration_ms": confirmation_report["short_duration_ms"],
    "confirmation_duration_ms": confirmation_report["confirmation_duration_ms"],
    "protocols": confirmation_report["protocols"],
})

## 6. Revisione visuale

Controllare che le soglie corrispondano a eventi plausibili osservando insieme voltaggio, calcio e componenti NMDA. Il notebook mostra i protocolli selezionati, i migliori candidati respinti e le conferme lunghe con il ritorno sotto soglia; la selezione automatica non sostituisce questa revisione.

In [ ]:
from IPython.display import Image, display

for figure_path in sorted(calibrator.figures_dir.glob("*.png")):
    print(figure_path.name)
    display(Image(filename=str(figure_path)))

## 7. Archivio e download diretto

Lo ZIP include configurazioni, sweep breve, conferma lunga, report di censura, tracce complete, grafici e stato teacher usato. Il download usa direttamente Base64 e un `Blob` nel browser.

In [ ]:
from pathlib import Path
from shutil import make_archive
from base64 import b64encode
from IPython.display import Javascript, display

artifact_dir = Path(calibrator.output_dir).resolve()
zip_base = Path("/kaggle/working/hayflow_dendritic_protocol_calibration")
zip_path = Path(make_archive(str(zip_base), "zip", root_dir=artifact_dir.parent, base_dir=artifact_dir.name))
print(f"ZIP creato: {zip_path}")
print(f"Dimensione: {zip_path.stat().st_size / 1024**2:.2f} MB")
encoded = b64encode(zip_path.read_bytes()).decode("ascii")
display(Javascript(f"""
(() => {{
    const binary = atob("{encoded}");
    const bytes = new Uint8Array(binary.length);
    for (let i = 0; i < binary.length; i++) {{
        bytes[i] = binary.charCodeAt(i);
    }}
    const blob = new Blob([bytes], {{type: "application/zip"}});
    const url = URL.createObjectURL(blob);
    const link = document.createElement("a");
    link.href = url;
    link.download = "hayflow_dendritic_protocol_calibration.zip";
    document.body.appendChild(link);
    link.click();
    link.remove();
    setTimeout(() => URL.revokeObjectURL(url), 10000);
}})();
"""))

## 8. Gate finale

Se il gate fallisce, scaricare comunque lo ZIP. Per lo sweep analizzare i grafici `best_rejected_*`; per la conferma controllare eventi censurati ed errore sull'overlap. Non modificare pesi o soglie per forzare artificialmente il risultato.

In [ ]:
assert calibration_report["valid"], (
    "Calibrazione incompleta; famiglie mancanti="
    + repr(calibration_report["missing_required_families"])
    + "; eventi mancanti="
    + repr(calibration_report["missing_required_event_kinds"])
)
assert confirmation_report["valid"], (
    "Conferma lunga incompleta: "
    + repr(confirmation_report["protocols"])
)